# v4.1 Adaptive Reasoning System — Demo & Inference

This notebook demonstrates the full reasoning control loop:

```
Problem → LLM parse → latent state → candidate reasoning actions
→ JEPA-style latent consequence prediction → EBM routing
→ specialized solver/tool execution → external verification
→ repair or memory update → repeat
```

**Requirements:** Upload the `v4_1_reasoning_system/` package to your Google Drive or install it directly.

In [ ]:
# Step 0: Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

import sys
# Adjust this path to where you uploaded the project on Google Drive
PROJECT_ROOT = '/content/drive/MyDrive/agi'
sys.path.insert(0, PROJECT_ROOT)

# Pin ortools<9.12 to avoid protobuf>=6 conflict with Colab's tensorflow/google-ai
!pip install -q torch transformers accelerate bitsandbytes "ortools>=9.7,<9.12" faiss-cpu pydantic rich numpy

In [ ]:
# Step 1: Import the system
import torch
from v4_1_reasoning_system.orchestration.controller import ReasoningController
from v4_1_reasoning_system.world_model.encoder import StateEncoder
from v4_1_reasoning_system.world_model.predictor import TransitionPredictor, ActionEncoder
from v4_1_reasoning_system.world_model.aux_heads import AuxiliaryHeads
from v4_1_reasoning_system.router.ebm_router import EBMRouter

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# Step 2: Initialize the controller (rule-based routing — no training needed)
LATENT_DIM = 128
ACTION_DIM = 32

controller = ReasoningController(
    parser=None,  # Will be created with defaults
    state_encoder=StateEncoder(latent_dim=LATENT_DIM).to(device),
    transition_predictor=TransitionPredictor(latent_dim=LATENT_DIM, action_dim=ACTION_DIM).to(device),
    action_encoder=ActionEncoder(action_dim=ACTION_DIM).to(device),
    aux_heads=AuxiliaryHeads(latent_dim=LATENT_DIM).to(device),
    ebm_router=EBMRouter(latent_dim=LATENT_DIM, action_dim=ACTION_DIM).to(device),
    use_learned_router=False,  # Start with rule-based
    max_iterations=5,
    max_time_seconds=120.0,
    device=device,
    latent_dim=LATENT_DIM,
    action_dim=ACTION_DIM,
)
print('Controller initialized (rule-based routing)')

## Example 1: Planning Task (Key-Door / Dependency Chain)

In [ ]:
planning_problem = "Process items key, door, chest in dependency order: get key before opening door, open door before accessing chest."

planning_parsed = {
    "domain": "planning",
    "description": "Key-door-chest dependency chain",
    "entities": ["key", "door", "chest"],
    "constraints": [
        {"name": "all_processed", "description": "All items must be processed", "ctype": "hard", "params": {}},
        {"name": "dep_order", "description": "Dependencies must be respected", "ctype": "hard", "params": {}},
    ],
    "objective": {"description": "Complete all items", "sense": "satisfy", "metric": "completeness", "params": {}},
    "ambiguities": [],
    "structured_data": {
        "subgoals": [
            {"name": "handle_key", "description": "Get the key", "dependencies": [], "params": {"entity": "key"}},
            {"name": "handle_door", "description": "Open the door", "dependencies": ["handle_key"], "params": {"entity": "door"}},
            {"name": "handle_chest", "description": "Access the chest", "dependencies": ["handle_door"], "params": {"entity": "chest"}},
        ]
    },
}

result = controller.solve_from_dict(planning_problem, planning_parsed)
print(f"Score: {result['score']:.3f}")
print(f"Valid: {result['valid']}")
print(f"Iterations: {result['iterations']}")
print(f"Solution: {result['solution']}")

## Example 2: Scheduling Task (Job Shop)

In [ ]:
scheduling_problem = "Schedule 4 jobs with precedence constraints to minimize makespan."

scheduling_parsed = {
    "domain": "scheduling",
    "description": "Schedule 4 jobs to minimize makespan",
    "entities": ["job_0", "job_1", "job_2", "job_3"],
    "constraints": [
        {"name": "precedence", "description": "Job dependencies must be respected", "ctype": "hard", "params": {}},
    ],
    "objective": {"description": "Minimize makespan", "sense": "minimize", "metric": "makespan", "params": {}},
    "ambiguities": [],
    "structured_data": {
        "type": "scheduling",
        "jobs": [
            {"name": "job_0", "duration": 3, "dependencies": []},
            {"name": "job_1", "duration": 5, "dependencies": ["job_0"]},
            {"name": "job_2", "duration": 2, "dependencies": ["job_0"]},
            {"name": "job_3", "duration": 4, "dependencies": ["job_1", "job_2"]},
        ],
        "horizon": 30,
        "resources": {},
    },
}

result = controller.solve_from_dict(scheduling_problem, scheduling_parsed)
print(f"Score: {result['score']:.3f}")
print(f"Valid: {result['valid']}")
print(f"Solution: {result['solution']}")

## Example 3: Optimization Task (Knapsack)

In [ ]:
optimization_problem = "Select items to maximize value within weight capacity 15."

optimization_parsed = {
    "domain": "optimization",
    "description": "Knapsack: maximize value with capacity 15",
    "entities": ["item_0", "item_1", "item_2", "item_3", "item_4"],
    "constraints": [
        {"name": "capacity", "description": "Total weight <= 15", "ctype": "hard",
         "params": {"type": "sum_leq", "bound": 15}},
    ],
    "objective": {"description": "Maximize total value", "sense": "maximize", "metric": "value", "params": {}},
    "ambiguities": [],
    "structured_data": {
        "type": "knapsack",
        "items": [
            {"name": "item_0", "weight": 5, "value": 10},
            {"name": "item_1", "weight": 4, "value": 8},
            {"name": "item_2", "weight": 6, "value": 12},
            {"name": "item_3", "weight": 3, "value": 7},
            {"name": "item_4", "weight": 7, "value": 15},
        ],
        "capacity": 15,
    },
}

result = controller.solve_from_dict(optimization_problem, optimization_parsed)
print(f"Score: {result['score']:.3f}")
print(f"Valid: {result['valid']}")
print(f"Solution: {result['solution']}")

## Inspect Reasoning Trajectory

In [ ]:
# Print detailed reasoning logs
for line in result['logs']:
    print(line)

In [ ]:
# Replay buffer statistics
stats = controller.replay_buffer.statistics()
print(f"Trajectories stored: {stats['count']}")
print(f"Success rate: {stats.get('success_rate', 0):.2%}")
print(f"Avg steps: {stats.get('avg_steps', 0):.1f}")
print(f"Mode counts: {stats.get('mode_counts', {})}")

## Save Checkpoint

In [ ]:
# Save everything to Google Drive
CHECKPOINT_DIR = '/content/drive/MyDrive/agi/checkpoints/v4_1_demo'
controller.save_checkpoint(CHECKPOINT_DIR)
print(f'Checkpoint saved to {CHECKPOINT_DIR}')